<a href="https://colab.research.google.com/github/Saleh-Furqan/OCR_playground/blob/benchmark-testing/tutorial_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

You need to adjust the code and set up google cloud credential yourself to run in local

#Speech-to-text

Install the dependencies. Example for Mac:
 - you need to install homebrew
 - pip install pyaudio
 - pip install requests
 - pip install --upgrade google-cloud-speech

In [ ]:
import pyaudio
import wave
import requests
import json

# Audio recording parameters
FORMAT = pyaudio.paInt16       # 16-bit depth
CHANNELS = 1                  # Mono channel
RATE = 16000                  # Sample rate 16kHz (must match API requirements)
CHUNK = 1024                  # Buffer size
RECORD_SECONDS = 5            # Recording duration (seconds)
OUTPUT_FILE = "output.wav"    # Output file name

# Initialize PyAudio
audio = pyaudio.PyAudio()

# Start recording
stream = audio.open(
    format=FORMAT,
    channels=CHANNELS,
    rate=RATE,
    input=True,
    frames_per_buffer=CHUNK
)
print("Recording started...")
frames = []
for _ in range(0, int(RATE / CHUNK * RECORD_SECONDS)):
    data = stream.read(CHUNK)
    frames.append(data)
print("Recording finished")

# Stop and close the stream
stream.stop_stream()
stream.close()
audio.terminate()

# Save the recorded audio to WAV file
with wave.open(OUTPUT_FILE, 'wb') as wf:
    wf.setnchannels(CHANNELS)
    wf.setsampwidth(audio.get_sample_size(FORMAT))
    wf.setframerate(RATE)
    wf.writeframes(b''.join(frames))

In [ ]:
#set credential

import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "./credential.json"

# Imports the Google Cloud client library


from google.cloud import speech



# def run_quickstart() -> speech.RecognizeResponse:
# Instantiates a client
client = speech.SpeechClient()

# The name of the audio file to transcribe
audio_file_path = "./output.wav"
with open(audio_file_path, "rb") as f:
    audio_content = f.read()

audio = speech.RecognitionAudio(content=audio_content)

config = speech.RecognitionConfig(
    encoding=speech.RecognitionConfig.AudioEncoding.LINEAR16,
    sample_rate_hertz=16000,
    language_code="en-US",
)

# Detects speech in the audio file
response = client.recognize(config=config, audio=audio)

for result in response.results:
    print(f"Transcript: {result.alternatives[0].transcript}")

## Naive Way

Record sound file -> Save file -> Upload to Google service -> Google process for you -> Receive feedback

### Latency

Without considering network latency, from the end of user's speech to receiving feedback from Google, you will need to wait for Google to process the entire file. This approach introduces significant delays because:

1. **Recording and Saving**: The entire audio file must be recorded and saved before processing begins.
2. **Uploading**: The complete file needs to be uploaded to the server, which can take time depending on the file size and network speed.
3. **Processing**: Google's service processes the entire file only after it has been fully uploaded.

This results in a **high end-to-end latency**, especially for long audio files.

## Streaming Approach

```plaintext
User starts speaking
    |
    v
Record audio chunk ----->Send chunk to server-->Server processes chunk-->Partial feedback
    |                                                                                   |
    v                                                                                   v
Record next audio chunk->Send chunk to server-->Process next chunk --> Update partial feedback
    |                                                                                   |
    v                                                                                   v
... (Repeat in parallel) ...                                           Update partial feedback
    |                                                                                   |
    v                                                                                   v
User finishes speaking--------------> Signal end of stream --------------> Final feedback
```

In contrast, the streaming approach sends audio data to the server in real-time chunks as the user speaks. This reduces latency significantly because:

1. **Real-Time Transmission**: Audio chunks are sent to the server immediately after being recorded, without waiting for the entire file to be completed.
2. **Parallel Processing**: The server can start processing each chunk as soon as it is received, overlapping computation with transmission.
3. **Incremental Feedback**: Partial results can be returned to the client while the user is still speaking, providing faster feedback.

### Latency Advantage

With streaming:
- The user receives partial results **while speaking**, reducing perceived latency.
- The total processing time is shorter because the server doesn't wait for the entire file to arrive before starting.




### Pseudo Code for Streaming

Refer to Official Documentation https://cloud.google.com/speech-to-text/docs/transcribe-streaming-audio?hl=zh-cn

```python

import queue
import re
import sys

from google.cloud import speech

import pyaudio

# Audio recording parameters
RATE = 16000
CHUNK = int(RATE / 10)  # 100ms


class MicrophoneStream:
    """Opens a recording stream as a generator yielding the audio chunks."""

    def __init__(self: object, rate: int = RATE, chunk: int = CHUNK) -> None:
        """The audio -- and generator -- is guaranteed to be on the main thread."""
        self._rate = rate
        self._chunk = chunk

        # Create a thread-safe buffer of audio data
        self._buff = queue.Queue()
        self.closed = True

    def __enter__(self: object) -> object:
        self._audio_interface = pyaudio.PyAudio()
        self._audio_stream = self._audio_interface.open(
            format=pyaudio.paInt16,
            # The API currently only supports 1-channel (mono) audio
            # https://goo.gl/z757pE
            channels=1,
            rate=self._rate,
            input=True,
            frames_per_buffer=self._chunk,
            # Run the audio stream asynchronously to fill the buffer object.
            # This is necessary so that the input device's buffer doesn't
            # overflow while the calling thread makes network requests, etc.
            stream_callback=self._fill_buffer,
        )

        self.closed = False

        return self

    def __exit__(
        self: object,
        type: object,
        value: object,
        traceback: object,
    ) -> None:
        """Closes the stream, regardless of whether the connection was lost or not."""
        self._audio_stream.stop_stream()
        self._audio_stream.close()
        self.closed = True
        # Signal the generator to terminate so that the client's
        # streaming_recognize method will not block the process termination.
        self._buff.put(None)
        self._audio_interface.terminate()

    def _fill_buffer(
        self: object,
        in_data: object,
        frame_count: int,
        time_info: object,
        status_flags: object,
    ) -> object:
        """Continuously collect data from the audio stream, into the buffer.

        Args:
            in_data: The audio data as a bytes object
            frame_count: The number of frames captured
            time_info: The time information
            status_flags: The status flags

        Returns:
            The audio data as a bytes object
        """
        self._buff.put(in_data)
        return None, pyaudio.paContinue

    def generator(self: object) -> object:
        """Generates audio chunks from the stream of audio data in chunks.

        Args:
            self: The MicrophoneStream object

        Returns:
            A generator that outputs audio chunks.
        """
        while not self.closed:
            # Use a blocking get() to ensure there's at least one chunk of
            # data, and stop iteration if the chunk is None, indicating the
            # end of the audio stream.
            chunk = self._buff.get()
            if chunk is None:
                return
            data = [chunk]

            # Now consume whatever other data's still buffered.
            while True:
                try:
                    chunk = self._buff.get(block=False)
                    if chunk is None:
                        return
                    data.append(chunk)
                except queue.Empty:
                    break

            yield b"".join(data)


def listen_print_loop(responses: object) -> str:
    """Iterates through server responses and prints them.

    The responses passed is a generator that will block until a response
    is provided by the server.

    Each response may contain multiple results, and each result may contain
    multiple alternatives; for details, see https://goo.gl/tjCPAU.  Here we
    print only the transcription for the top alternative of the top result.

    In this case, responses are provided for interim results as well. If the
    response is an interim one, print a line feed at the end of it, to allow
    the next result to overwrite it, until the response is a final one. For the
    final one, print a newline to preserve the finalized transcription.

    Args:
        responses: List of server responses

    Returns:
        The transcribed text.
    """
    num_chars_printed = 0
    for response in responses:
        if not response.results:
            continue

        # The `results` list is consecutive. For streaming, we only care about
        # the first result being considered, since once it's `is_final`, it
        # moves on to considering the next utterance.
        result = response.results[0]
        if not result.alternatives:
            continue

        # Display the transcription of the top alternative.
        transcript = result.alternatives[0].transcript

        # Display interim results, but with a carriage return at the end of the
        # line, so subsequent lines will overwrite them.
        #
        # If the previous result was longer than this one, we need to print
        # some extra spaces to overwrite the previous result
        overwrite_chars = " " * (num_chars_printed - len(transcript))

        if not result.is_final:
            sys.stdout.write(transcript + overwrite_chars + "\r")
            sys.stdout.flush()

            num_chars_printed = len(transcript)

        else:
            print(transcript + overwrite_chars)

            # Exit recognition if any of the transcribed phrases could be
            # one of our keywords.
            if re.search(r"\b(exit|quit)\b", transcript, re.I):
                print("Exiting..")
                break

            num_chars_printed = 0

    return transcript


def main() -> None:
    """Transcribe speech from audio file."""
    # See http://g.co/cloud/speech/docs/languages
    # for a list of supported languages.
    language_code = "en-US"  # a BCP-47 language tag

    client = speech.SpeechClient()
    config = speech.RecognitionConfig(
        encoding=speech.RecognitionConfig.AudioEncoding.LINEAR16,
        sample_rate_hertz=RATE,
        language_code=language_code,
    )

    streaming_config = speech.StreamingRecognitionConfig(
        config=config, interim_results=True
    )

    with MicrophoneStream(RATE, CHUNK) as stream:
        audio_generator = stream.generator()
        requests = (
            speech.StreamingRecognizeRequest(audio_content=content)
            for content in audio_generator
        )

        responses = client.streaming_recognize(streaming_config, requests)

        # Now, put the transcription responses to use.
        listen_print_loop(responses)


if __name__ == "__main__":
    main()
```

# Text-to-speech
Install the dependencies.
 - pip install google-cloud-texttospeech

In [ ]:
#set credential

import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "./credential.json"


"""Synthesizes speech from the input string of text."""
from google.cloud import texttospeech

text = "Hello there. Could you help set my mac to dark mode?"
client = texttospeech.TextToSpeechClient()

input_text = texttospeech.SynthesisInput(text=text)

# Note: the voice can also be specified by name.
# Names of voices can be retrieved with client.list_voices().
voice = texttospeech.VoiceSelectionParams(
    language_code="en-US",
    name="en-US-Standard-C",
    ssml_gender=texttospeech.SsmlVoiceGender.FEMALE,
)

audio_config = texttospeech.AudioConfig(
    audio_encoding=texttospeech.AudioEncoding.MP3
)

response = client.synthesize_speech(
    request={"input": input_text, "voice": voice, "audio_config": audio_config}
)

# The response's audio_content is binary.
with open("output.mp3", "wb") as out:
    out.write(response.audio_content)
    print('Audio content written to file "output.mp3"')



### **Naive Way**  
**Process Flow**:  
```plaintext  
LLM generates full text -> Save text to file -> Send to TTS service -> TTS generates full audio -> Play audio  
```  

#### **Latency Analysis**  
1. **Wait for LLM completion**: The entire text must be generated by the LLM before TTS can start.  
2. **Full-text processing**: The TTS service processes the entire text at once.  
3. **End-to-end delay**: The user waits for both LLM generation and TTS processing to finish before hearing any audio.  

**Example Scenario**:  
- LLM generates a 10-sentence response in 5 seconds.  
- TTS processes the entire text in 3 seconds.  
- **Total latency**: 8 seconds before audio playback starts.  

---

### **Streaming Approach**  
**Process Flow**:  
```plaintext  
LLM generates text sentence-by-sentence --> Stream each sentence to TTS --> TTS generates audio chunk --> Play audio chunk  
```  

#### **Latency Advantage**  
1. **Parallel processing**:  
   - LLM generates the next sentence while TTS processes the current one.  
   - Audio playback starts as soon as the first chunk is ready.  
2. **Incremental feedback**:  
   - Users hear partial results (e.g., first sentence) while the LLM is still generating后续内容.  

**Example Scenario**:  
- LLM generates the first sentence in 0.5 seconds.  
- TTS processes and plays the first sentence in 0.3 seconds.  
- **Total latency**: 0.8 seconds for the first audio chunk, with后续 chunks following incrementally.  

---

### **Pseudo Code for Streaming**  
```python  
import threading
import queue
import time

# 1. LLM Generation Thread (Producer)
def llm_generate_stream(prompt, output_queue):
    for sentence in ["Hello!", "How are you?", "This is a streaming example."]:
        time.sleep(0.5)  # Simulate LLM generation delay
        output_queue.put(sentence)  # Send sentence to TTS queue
    output_queue.put(None)  # Signal end of generation

# 2. TTS Processing Thread (Consumer/Producer)
def tts_processor(input_queue, audio_queue):
    while True:
        sentence = input_queue.get()
        if sentence is None:  # End signal
            audio_queue.put(None)
            break
        audio_chunk = tts_service.convert_text_to_audio(sentence)  # Simulate TTS
        audio_queue.put(audio_chunk)

# 3. Audio Playback Thread (Consumer)
def audio_player(audio_queue):
    while True:
        audio_chunk = audio_queue.get()
        if audio_chunk is None:  # End signal
            break
        audio_player.play(audio_chunk)  # Play audio immediately

# 4. Full Workflow with Concurrency
prompt = "Start the conversation."

# Create queues for inter-thread communication
llm_to_tts_queue = queue.Queue()
tts_to_audio_queue = queue.Queue()

# Start threads
llm_thread = threading.Thread(target=llm_generate_stream, args=(prompt, llm_to_tts_queue))
tts_thread = threading.Thread(target=tts_processor, args=(llm_to_tts_queue, tts_to_audio_queue))
audio_thread = threading.Thread(target=audio_player, args=(tts_to_audio_queue,))

llm_thread.start()
tts_thread.start()
audio_thread.start()

# Wait for all threads to finish
llm_thread.join()
tts_thread.join()
audio_thread.join()
```  